<a href="https://colab.research.google.com/github/ryukf333/fruit-freshness-ai/blob/main/01Data_Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
import shutil
from pathlib import Path
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Path to zip
drive_raw_dir = Path('/content/drive/MyDrive/fruit_freshness_project/data/raw/')
zip_files = list(drive_raw_dir.glob('*.zip'))

if not zip_files:
    raise FileNotFoundError(f"No .zip file found in {drive_raw_dir}.")

zip_path = zip_files[0]
print(f"Found dataset archive: {zip_path.name}")

# 3. Target local directory on Colab SSD
local_raw_dir = Path('/content/dataset_raw')

# 4. Extract archive locally
if not local_raw_dir.exists():
    print("Unzipping images onto Colab's high-speed local disk...")
    with zipfile.ZipFile(zip_path, 'r') as archive:
        archive.extractall(local_raw_dir)
    print("Extraction finished successfully!")
else:
    print("Dataset already unpacked on local disk.")


Mounted at /content/drive
Found dataset archive: AgriFreshNET.zip
Unzipping images onto Colab's high-speed local disk...
Extraction finished successfully!


In [ ]:
# Inspect the extracted directory layout
extracted_folders = sorted([p for p in local_raw_dir.rglob('*') if p.is_dir()])

print(f"Total subdirectories created: {len(extracted_folders)}")
print("\nFirst 15 directories inside your dataset:")
for folder in extracted_folders[:]:
    # Print relative path from the extraction root
    print("  ->", folder.relative_to(local_raw_dir))

Total subdirectories created: 25

First 15 directories inside your dataset:
  -> Processed Data
  -> Processed Data/Fresh Banana(1-4)
  -> Processed Data/Fresh Bittermelon(1-3)
  -> Processed Data/Fresh Cucumber(1-6)
  -> Processed Data/Fresh Orange(1-9)
  -> Processed Data/Fresh Papaya(1-4)
  -> Processed Data/Fresh Tomato(1-10)
  -> Processed Data/Fresh eggplant(1-4)
  -> Processed Data/Fresh pineapple(1-15)
  -> Processed Data/Rotten Bittermelon(5-8)
  -> Processed Data/Rotten Cucumber(12-20)
  -> Processed Data/Rotten Orange(20-35)
  -> Processed Data/Rotten Papaya(7-12)
  -> Processed Data/Rotten Pineapple(25-35)
  -> Processed Data/Rotten Tomato(24-35)
  -> Processed Data/Rotten banana(7-13)
  -> Processed Data/Rotten eggplant(8-15)
  -> Processed Data/Semi Fresh Bittermelon ( 3-5)
  -> Processed Data/Semi Fresh Cucumber(6-12)
  -> Processed Data/Semi Fresh Papaya(4-7)
  -> Processed Data/Semi fresh Orange(9-20)
  -> Processed Data/Semi fresh Pineapple (15-25)
  -> Processed Data

In [ ]:
import os
import re
import hashlib
from pathlib import Path
from PIL import Image, ImageFile
import pandas as pd
from tqdm import tqdm

# Prevent PIL crashes on minor truncation warnings
ImageFile.LOAD_TRUNCATED_IMAGES = False

LOCAL_RAW_ROOT = Path('/content/dataset_raw')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/fruit_freshness_project')

def parse_folder_attributes(folder_name: str):

    # 1. Extract shelf-life interval within parentheses
    shelf_life_match = re.search(r'\((.*?)\)', folder_name)
    shelf_life = shelf_life_match.group(1).strip() if shelf_life_match else "Unknown"

    # Remove the bracketed interval to isolate the names
    name_only = re.sub(r'\(.*?\)', '', folder_name).strip()
    name_normalized = name_only.lower().replace('_', ' ')

    # 2. Determine Freshness Class
    if 'semi' in name_normalized:
        freshness = 'Semi-Fresh'
    elif 'rotten' in name_normalized:
        freshness = 'Rotten'
    elif 'fresh' in name_normalized:
        freshness = 'Fresh'
    else:
        freshness = 'Unknown'

    # 3. Extract Produce Type by removing freshness keywords
    produce_str = re.sub(r'(?i)(semi\s*fresh|semi_fresh|rotten|fresh)', '', name_only).strip()
    produce = produce_str.capitalize() if produce_str else "Unknown"

    return freshness, produce, shelf_life

def compute_file_md5(file_path: Path, chunk_size: int = 65536) -> str:
    """Computes MD5 hash to detect exact byte-level image duplicates."""
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        while chunk := f.read(chunk_size):
            hasher.update(chunk)
    return hasher.hexdigest()

# Crawl and audit all image files
supported_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
image_paths = [p for p in LOCAL_RAW_ROOT.rglob('*') if p.suffix.lower() in supported_extensions]

print(f"Discovered {len(image_paths)} total images. Parsing metadata and auditing integrity...")

records = []
seen_hashes = {}
corrupted_count = 0

for img_path in tqdm(image_paths):
    # Verify image readability and get spatial dimensions
    try:
        with Image.open(img_path) as img:
            img.verify()
            width, height = img.size
            channels = len(img.getbands())
            format_type = img.format
    except Exception:
        corrupted_count += 1
        continue

    # Identify duplicates
    file_hash = compute_file_md5(img_path)
    is_duplicate = file_hash in seen_hashes
    duplicate_of = seen_hashes.get(file_hash, None)

    if not is_duplicate:
        seen_hashes[file_hash] = str(img_path)

    # Parse metadata from parent folder name
    parent_folder = img_path.parent.name
    freshness, produce, shelf_life = parse_folder_attributes(parent_folder)

    records.append({
        'image_id': img_path.stem,
        'file_name': img_path.name,
        'file_path': str(img_path.resolve()),
        'produce_type': produce,
        'freshness_label': freshness,
        'shelf_life_days': shelf_life,
        'width': width,
        'height': height,
        'channels': channels,
        'format': format_type,
        'file_size_kb': round(img_path.stat().st_size / 1024, 2),
        'md5_hash': file_hash,
        'is_duplicate': is_duplicate,
        'duplicate_of': duplicate_of
    })

# Convert to DataFrame
df_all = pd.DataFrame(records)

# Filter out duplicates for modeling
df_manifest = df_all[~df_all['is_duplicate']].reset_index(drop=True)

# Save to Google Drive
output_csv = DRIVE_PROJECT_ROOT / 'metadata/dataset_manifest.csv'
output_csv.parent.mkdir(parents=True, exist_ok=True)  # <-- Add this line
df_manifest.to_csv(output_csv, index=False)

print(f"\nAudit complete!")
print(f"  • Valid unique images: {len(df_manifest)}")
print(f"  • Duplicates flagged: {df_all['is_duplicate'].sum()}")
print(f"  • Corrupted files skipped: {corrupted_count}")
print(f"  • Manifest saved to: {output_csv}")

Discovered 14160 total images. Parsing metadata and auditing integrity...


100%|██████████| 14160/14160 [00:04<00:00, 3130.91it/s]



Audit complete!
  • Valid unique images: 14120
  • Duplicates flagged: 40
  • Corrupted files skipped: 0
  • Manifest saved to: /content/drive/MyDrive/fruit_freshness_project/metadata/dataset_manifest.csv


In [ ]:
# 1. Freshness Stage Distribution
print("--- Freshness Stage Distribution ---")
print(df_manifest['freshness_label'].value_counts())

# 2. Produce Type Breakdown
print("\n--- Produce Types Discovered ---")
print(df_manifest['produce_type'].value_counts())

# 3. Produce by Freshness Cross-Tabulation
print("\n--- Produce vs. Freshness Matrix ---")
cross_tab = pd.crosstab(df_manifest['produce_type'], df_manifest['freshness_label'], margins=True)
display(cross_tab)

--- Freshness Stage Distribution ---
freshness_label
Rotten        4717
Fresh         4710
Semi-Fresh    4693
Name: count, dtype: int64

--- Produce Types Discovered ---
produce_type
Eggplant       1770
Cucumber       1769
Pineapple      1769
Tomato         1768
Bittermelon    1767
Papaya         1766
Banana         1762
Orange         1749
Name: count, dtype: int64

--- Produce vs. Freshness Matrix ---


freshness_label,Fresh,Rotten,Semi-Fresh,All
produce_type,,,,
Banana,584,590,588,1762
Bittermelon,590,589,588,1767
Cucumber,589,590,590,1769
Eggplant,590,590,590,1770
Orange,590,588,571,1749
Papaya,587,590,589,1766
Pineapple,590,590,589,1769
Tomato,590,590,588,1768
All,4710,4717,4693,14120


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/fruit_freshness_project')
manifest_path = DRIVE_PROJECT_ROOT / 'metadata/dataset_manifest.csv'

# 1. Load the cataloged manifest
df = pd.read_csv(manifest_path)

# 2. Create a compound stratification key
df['strat_key'] = df['produce_type'] + '_' + df['freshness_label']

# 3. First split: 70% Train, 30% Temporary (Validation + Test)
df_train, df_temp = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df['strat_key']
)

# 4. Second split: Divide temporary pool equally into 15% Val and 15% Test
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp['strat_key']
)

# 5. Clean up the temporary column
for split_df in [df_train, df_val, df_test]:
    split_df.drop(columns=['strat_key'], inplace=True)

# 6. Save frozen splits to Google Drive
splits_dir = DRIVE_PROJECT_ROOT / 'data/processed'
splits_dir.mkdir(parents=True, exist_ok=True)

train_path = splits_dir / 'train_manifest.csv'
val_path = splits_dir / 'val_manifest.csv'
test_path = splits_dir / 'test_manifest.csv'

df_train.to_csv(train_path, index=False)
df_val.to_csv(val_path, index=False)
df_test.to_csv(test_path, index=False)

print(f"Splits saved successfully in: {splits_dir}")
print(f"  • Training set:   {len(df_train):,} samples ({len(df_train)/len(df)*100:.1f}%)")
print(f"  • Validation set: {len(df_val):,} samples ({len(df_val)/len(df)*100:.1f}%)")
print(f"  • Test set:       {len(df_test):,} samples ({len(df_test)/len(df)*100:.1f}%)")

Splits saved successfully in: /content/drive/MyDrive/fruit_freshness_project/data/processed
  • Training set:   9,884 samples (70.0%)
  • Validation set: 2,118 samples (15.0%)
  • Test set:       2,118 samples (15.0%)


In [ ]:
# Verify class proportions across all three splits
print("--- Class Proportions (%) Across Splits ---")
prop_summary = pd.DataFrame({
    'Train (%)': df_train['freshness_label'].value_counts(normalize=True) * 100,
    'Val (%)': df_val['freshness_label'].value_counts(normalize=True) * 100,
    'Test (%)': df_test['freshness_label'].value_counts(normalize=True) * 100
})
display(prop_summary.round(2))

--- Class Proportions (%) Across Splits ---


,Train (%),Val (%),Test (%)
freshness_label,,,
Fresh,33.36,33.33,33.38
Rotten,33.40,33.52,33.33
Semi-Fresh,33.25,33.14,33.29
